# SNJ Product Retrieval — Constraints & Ranking / Truy hồi và ràng buộc

**Trương Minh Hoàng · MSSV 25410008**

Notebook này trình bày bản mã được tách từ bài Hybrid RAG và sửa các điều kiện bắt buộc.
Catalog đi kèm gồm **12 sản phẩm hoàn toàn giả lập**, không phản ánh giá/tồn kho SNJ.
Mặc định chạy **TF-IDF + rules**, không gọi LLM. Đặt `SNJ_DENSE_MODEL_DIR` để dùng model MiniLM đã tải về;
khi đó output được ghi rõ `dense_lexical_rules`. Không tự thay model khi thiếu trọng số.

The original dense-retrieval coursework is preserved under `research/`.
Passing these software checks does not establish the historical 100% retrieval claim on new customers.

In [1]:
from pathlib import Path
import os, sys, json
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from retrieval import CatalogRetriever,format_answer,matches
from domain import parse_query,extract_query_sizes
catalog=pd.read_csv(ROOT/'data/sample_catalog.csv')
engine=CatalogRetriever(catalog,model_dir=os.environ.get('SNJ_DENSE_MODEL_DIR'))
print('Fictional products:',len(catalog),'mode:',engine.mode)
display(catalog[['product_id','name','category','price','size_value']])


Fictional products: 12 mode: lexical_rules


,product_id,name,category,price,size_value
0,DEMO-001,Cặp đá ruby rời 1 ct,loose_stone,850000.0,NaN
1,DEMO-002,Nhẫn bạc nữ đá ruby size 16,ring,1200000.0,16.0
2,DEMO-003,Lắc tay đá citrine vàng 8ly,bracelet,600000.0,8.0
3,DEMO-004,Lắc tay đá citrine vàng 8ly5,bracelet,650000.0,8.5
4,DEMO-005,Lắc tay đá citrine vàng 18ly,bracelet,750000.0,18.0
5,DEMO-006,Vòng cẩm thạch size 52 kiểm định PNJ,jade_bangle,1800000.0,52.0
6,DEMO-007,Vòng cẩm thạch size 57,jade_bangle,NaN,57.0
7,DEMO-008,Lắc tay đá mắt hổ nâu 10ly,bracelet,450000.0,10.0
8,DEMO-009,Bông tai bạc sapphire xanh,earrings,800000.0,NaN
9,DEMO-010,Mặt dây thạch anh tóc vàng,pendant,500000.0,NaN


## 1. Hiểu loại sản phẩm / Product intent
Đá rời dùng để chế tác được giữ riêng với nhẫn/bông tai thành phẩm. Các yêu cầu đơn giản về đá, màu, giới tính, kích thước và ngân sách được chuyển thành thuộc tính có thể kiểm tra.

In [2]:
queries=['ruby rời làm bông tai dưới 1 triệu', 'lắc tay đá citrine vàng 8ly5',
 'vòng cẩm thạch size 52 kiểm định PNJ dưới 2 triệu','nhẫn nam garnet','nhẫn sapphire']
display(pd.DataFrame([{'query':q,**parse_query(q)} for q in queries]))


,query,product_intent,stones,colors,genders,sizes,max_price,certificate_required,earring_purpose,ring_or_pendant_purpose
0,ruby rời làm bông tai dưới 1 triệu,loose_stone,[ruby],[],[],[],1000000.0,False,True,False
1,lắc tay đá citrine vàng 8ly5,stone_bracelet,[citrine],[vàng],[],[8.5],NaN,False,False,False
2,vòng cẩm thạch size 52 kiểm định PNJ dưới 2 triệu,jade_bangle,[jade],[],[],[52.0],2000000.0,True,False,False
3,nhẫn nam garnet,ring,[garnet],[],[male],[],NaN,False,False,False
4,nhẫn sapphire,ring,[sapphire],[],[],[],NaN,False,False,False


## 2. Lọc cứng trước khi xếp hạng / Filter before ranking
Giá chưa biết không thỏa điều kiện ngân sách. Không nới điều kiện khi không có kết quả. So sánh kích thước theo số: `8ly`, `8ly5` và `18ly` khác nhau.

In [3]:
examples=[]
for q in queries:
    result=engine.retrieve(q)
    examples.append({'query':q,'count':len(result),'product_ids':', '.join(result.product_id),'mode':engine.mode})
display(pd.DataFrame(examples))
assert engine.retrieve('lắc tay citrine 8ly5').product_id.tolist()==['DEMO-004']
assert engine.retrieve('ruby rời dưới 1 triệu').product_id.tolist()==['DEMO-001']
assert engine.retrieve('nhẫn sapphire').empty


,query,count,product_ids,mode
0,ruby rời làm bông tai dưới 1 triệu,1,DEMO-001,lexical_rules
1,lắc tay đá citrine vàng 8ly5,1,DEMO-004,lexical_rules
2,vòng cẩm thạch size 52 kiểm định PNJ dưới 2 triệu,1,DEMO-006,lexical_rules
3,nhẫn nam garnet,1,DEMO-012,lexical_rules
4,nhẫn sapphire,0,,lexical_rules


In [4]:
result=engine.retrieve('ruby rời làm bông tai dưới 1 triệu')
print(format_answer(result))
display(result[['product_id','dense_score','keyword_score','rule_score','final_score','retrieval_mode']])


DEMO-001 — Cặp đá ruby rời 1 ct — 850,000 VND
Thông tin lấy từ bản catalog đã nạp; giá và tồn kho thực tế cần được kiểm tra tại nguồn.


,product_id,dense_score,keyword_score,rule_score,final_score,retrieval_mode
0,DEMO-001,0.0,0.262547,0.5,0.15,lexical_rules


## 3. Kiểm tra bộ tình huống / Scenario checks
Các đáp án dưới đây được gán theo catalog giả lập, có ca đúng loại hàng, giá thiếu, size thập phân, chứng thư và không có kết quả. Chúng kiểm tra hợp đồng phần mềm, không phải ước lượng độ chính xác ngoài tập.

In [5]:
cases=[('ruby rời dưới 1 triệu',{'DEMO-001'}),('lắc tay citrine 8ly',{'DEMO-003'}),
 ('lắc tay citrine 8ly5',{'DEMO-004'}),('vòng cẩm thạch size 52 kiểm định PNJ',{'DEMO-006'}),
 ('nhẫn nam garnet',{'DEMO-012'}),('nhẫn sapphire',set())]
checks=[]
for query,expected in cases:
    got=set(engine.retrieve(query,top_k=20).product_id)
    checks.append({'query':query,'expected':sorted(expected),'actual':sorted(got),'passed':got==expected})
assert all(r['passed'] for r in checks)
display(pd.DataFrame(checks))


,query,expected,actual,passed
0,ruby rời dưới 1 triệu,[DEMO-001],[DEMO-001],True
1,lắc tay citrine 8ly,[DEMO-003],[DEMO-003],True
2,lắc tay citrine 8ly5,[DEMO-004],[DEMO-004],True
3,vòng cẩm thạch size 52 kiểm định PNJ,[DEMO-006],[DEMO-006],True
4,nhẫn nam garnet,[DEMO-012],[DEMO-012],True
5,nhẫn sapphire,[],[],True


## 4. Chạy với catalog riêng / Local catalog
Set `SNJ_CATALOG_PATH` để kiểm tra schema/hash của catalog gốc. Cell chỉ xuất tổng số và số giá thiếu; không in bản ghi thương mại. Dense evaluation, bộ query độc lập và tích hợp WooCommerce được theo dõi riêng.

In [6]:
import hashlib
catalog_audit={'status':'NOT_SUPPLIED','source':'fictional sample only'}
if os.environ.get('SNJ_CATALOG_PATH'):
    p=Path(os.environ['SNJ_CATALOG_PATH']);raw=p.read_bytes();actual=pd.read_csv(p)
    price=pd.to_numeric(actual.price,errors='coerce')
    catalog_audit={'status':'SCHEMA_AUDIT','rows':len(actual),'columns':len(actual.columns),
       'sha256':hashlib.sha256(raw).hexdigest(),'unique_product_ids':int(actual.product_id.nunique()),
       'duplicate_product_id_rows':int(actual.product_id.duplicated().sum()),
       'missing_or_nonpositive_prices':int((price.isna()|price.le(0)).sum())}
    catalog_audit['status']='REQUIRES_ID_REVIEW' if catalog_audit['duplicate_product_id_rows'] else 'SCHEMA_PASS'
display(pd.Series(catalog_audit))


status                                                          REQUIRES_ID_REVIEW
rows                                                                           376
columns                                                                         22
sha256                           408df070d1979cc430e5f6c47f50f075d7590696d8cc42...
unique_product_ids                                                             371
duplicate_product_id_rows                                                        5
missing_or_nonpositive_prices                                                    0
dtype: object

In [7]:
LAB_RESULT={'status':'PASS','scope':'synthetic catalog constraint checks','mode':engine.mode,
 'fictional_catalog_rows':len(catalog),'scenario_checks':checks,'local_catalog_audit':catalog_audit,
 'generation':'NOT_RUN','live_commerce_api':'NOT_RUN',
 'fresh_dense_evaluation':'PASS' if engine.mode=='dense_lexical_rules' else 'NOT_RUN'}
(ROOT/'evidence').mkdir(exist_ok=True)
(ROOT/'evidence/current_checks.json').write_text(json.dumps(LAB_RESULT,indent=2,ensure_ascii=False)+'\n')
print('Saved evidence/current_checks.json')


Saved evidence/current_checks.json
